# Этап 2. Очистка данных

Источник для очистки - Parquet с типизированными данными после первого этапа.

In [31]:
from pathlib import Path

import duckdb

cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd
parquet_path = BASE_DIR / "data/processed/raw_typed.parquet"

con = duckdb.connect()
con.read_parquet(str(parquet_path), file_row_number=True).create_view("cars_raw");

In [36]:
con.execute("PRAGMA disable_progress_bar");

## 2.1. Дубли объявлений

Для каждой ссылки остаётся запись с самой свежей датой сбора. При равных датах предпочтение у первой строки в Parquet. Строки без ссылки сохраняются.

In [32]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW cars_dedup AS
    SELECT * EXCLUDE (file_row_number)
    FROM cars_raw
    QUALIFY NULLIF(TRIM(link), '') IS NULL
        OR ROW_NUMBER() OVER (
            PARTITION BY link
            ORDER BY parse_date DESC NULLS LAST, file_row_number
        ) = 1
""")

rows_before = con.execute("SELECT COUNT(*) FROM cars_raw").fetchone()[0]
rows_after = con.execute("SELECT COUNT(*) FROM cars_dedup").fetchone()[0]
print(f"Строк до очистки: {rows_before:,}\nСтрок после очистки: {rows_after:,}\nУдалено дублей: {rows_before - rows_after:,}")

Строк до очистки: 1,294,757
Строк после очистки: 950,457
Удалено дублей: 344,300


## 2.2. Выбросы цены

Для каждой пары марка-модель с минимум 100 положительными ценами задаются границы: 1-й и 99-й процентили. Цены на границах сохраняются. Порог в 100 записей, чтобы каждому проценту соответствовало хотя бы одно объявление. Для малых групп и строк без марки или модели сохраняются все положительные цены. Нулевые и отрицательные цены исключаются. Пропуски цены остаются до пункта 2.5.

In [33]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW cars_price_clean AS

    WITH price_bounds AS (
        SELECT brand, name,
               quantile_disc(price, 0.01) AS lower_price,
               quantile_disc(price, 0.99) AS upper_price
        FROM cars_dedup
        WHERE price > 0 AND TRIM(brand) <> '' AND TRIM(name) <> ''
        GROUP BY brand, name
        HAVING COUNT(*) >= 100
    )
    
    SELECT cars.*
    FROM cars_dedup AS cars
    LEFT JOIN price_bounds AS bounds USING (brand, name)
    WHERE price IS NULL
       OR (price > 0 AND (lower_price IS NULL OR price BETWEEN lower_price AND upper_price))
""")

In [38]:
con.sql("SELECT * EXCLUDE (description) FROM cars_price_clean ORDER BY brand, name LIMIT 10").show()

┌─────────┬─────────┬────────────┬─────────────┬──────────┬───────┬─────────┬──────────────┬───────┬─────────┬──────────────────────────────────┬────────────┬────────────────────┬─────────────────────┬────────────────┬────────────────────────────────────────────────────────┬─────────────────────┐
│  brand  │  name   │  bodyType  │    color    │ fuelType │ year  │ mileage │ transmission │ power │  price  │       vehicleConfiguration       │ engineName │ engineDisplacement │        date         │    location    │                          link                          │     parse_date      │
│ varchar │ varchar │  varchar   │   varchar   │ varchar  │ int16 │  int32  │   varchar    │ int16 │  int64  │             varchar              │  varchar   │      varchar       │      timestamp      │    varchar     │                        varchar                         │      timestamp      │
├─────────┼─────────┼────────────┼─────────────┼──────────┼───────┼─────────┼──────────────┼───────┼──────

In [35]:
rows_after_price = con.execute("SELECT COUNT(*) FROM cars_price_clean").fetchone()[0]
print(f"Строк до фильтрации цен: {rows_after:,}")
print(f"Исключено по цене: {rows_after - rows_after_price:,}")
print(f"Строк после фильтрации: {rows_after_price:,}")

Строк до фильтрации цен: 950,457
Исключено по цене: 16,167
Строк после фильтрации: 934,290


## 2.3. Год, пробег и мощность

Допустимые значения (включительно): год выпуска от 1960 до 2023, мощность от 20 до 1000 л.с., пробег от 0 км.

Для объявлений, опубликованных в год выпуска, приведена сводка пробега. Автомобили с нулевым пробегом в более поздний год и объявления, опубликованные до года выпуска, учтены отдельно и сохранены для проверки.

In [48]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW cars_specs_clean AS
    SELECT *
    FROM cars_price_clean
    WHERE (year IS NULL OR year BETWEEN 1960 AND 2023)
      AND (power IS NULL OR power BETWEEN 20 AND 1000)
      AND (mileage IS NULL OR mileage >= 0)
""")

rows_after_specs = con.execute("SELECT COUNT(*) FROM cars_specs_clean").fetchone()[0]
print(f"Исключено по году, пробегу или мощности: {rows_after_price - rows_after_specs:,}")
print(f"Строк после фильтрации: {rows_after_specs:,}")

Исключено по году, пробегу или мощности: 399
Строк после фильтрации: 933,891


In [ ]:
con.sql("""
    SELECT
        COUNT(*) AS "Объявления в год выпуска",
        COUNT(mileage) AS "С указанным пробегом",
        MEDIAN(mileage) AS "Медиана, км",
        MAX(mileage) AS "Максимум, км"
    FROM cars_specs_clean
    WHERE YEAR(date) = year
""").show()

┌──────────────────────────┬──────────────────────┬─────────────┬──────────────┐
│ Объявления в год выпуска │ С указанным пробегом │ Медиана, км │ Максимум, км │
│          int64           │        int64         │   double    │    int32     │
├──────────────────────────┼──────────────────────┼─────────────┼──────────────┤
│                     9270 │                   55 │      1000.0 │         5000 │
└──────────────────────────┴──────────────────────┴─────────────┴──────────────┘



## 2.4. Нормализация марок и моделей

Лишние пробелы удаляются. Для вариантов, различающихся только регистром, используется самое частое написание. Для моделей частота определяется внутри марки. При равной частоте выбирается первое написание в порядке сортировки.

In [52]:
con.execute(r"""
    CREATE OR REPLACE TEMP VIEW cars_categories_clean AS
    WITH trimmed AS (
        SELECT * REPLACE (
            trim(regexp_replace(brand, '\s+', ' ', 'g')) AS brand,
            trim(regexp_replace(name, '\s+', ' ', 'g')) AS name
        )
        FROM cars_specs_clean
    )
    SELECT * REPLACE (
        mode(brand ORDER BY brand) OVER (PARTITION BY lower(brand)) AS brand,
        mode(name ORDER BY name) OVER (PARTITION BY lower(brand), lower(name)) AS name
    )
    FROM trimmed
""")

In [54]:
con.sql("""SELECT * EXCLUDE (description) FROM cars_categories_clean LIMIT 10""").show()

┌────────────┬─────────┬───────────┬─────────────┬──────────┬───────┬─────────┬──────────────┬───────┬─────────┬─────────────────────────────┬────────────┬────────────────────┬─────────────────────┬─────────────────┬───────────────────────────────────────────────────────────────┬─────────────────────┐
│   brand    │  name   │ bodyType  │    color    │ fuelType │ year  │ mileage │ transmission │ power │  price  │    vehicleConfiguration     │ engineName │ engineDisplacement │        date         │    location     │                             link                              │     parse_date      │
│  varchar   │ varchar │  varchar  │   varchar   │ varchar  │ int16 │  int32  │   varchar    │ int16 │  int64  │           varchar           │  varchar   │      varchar       │      timestamp      │     varchar     │                            varchar                            │      timestamp      │
├────────────┼─────────┼───────────┼─────────────┼──────────┼───────┼─────────┼────────────

## 2.5. Пропуски

Для анализа обязательны цена, год выпуска и марка. Пустая марка после нормализации также считается пропуском. Столбцы `vehicleConfiguration`, `engineName` и `engineDisplacement` исключены из аналитической выборки из-за высокой доли пропусков. Пропуски в остальных полях сохраняются.

In [58]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW cars_missing_clean AS
    SELECT * EXCLUDE (vehicleConfiguration, engineName, engineDisplacement)
    FROM cars_categories_clean
    WHERE price IS NOT NULL AND year IS NOT NULL AND brand IS NOT NULL
""")

rows_after_missing = con.execute("SELECT COUNT(*) FROM cars_missing_clean").fetchone()[0]
print(f"Исключено без цены, года или марки: {rows_after_specs - rows_after_missing:,}")
print(f"Осталось объявлений: {rows_after_missing:,}")

Исключено без цены, года или марки: 412,409
Осталось объявлений: 521,482


## 2.6. Новые признаки

Возраст автомобиля рассчитан на 2023 год. Среднегодовой пробег округлён до целых километров. Для машин 2023 года и объявлений без пробега значение отсутствует.

Происхождение определяется по марке. К отечественным отнесены российские марки и их советские предшественники. Для категории «Прочие авто» происхождение неизвестно. Ценовые сегменты для этой работы заданы в рублях: бюджетный — до 1 млн, средний — от 1 млн до 3 млн, премиум — от 3 млн. Нижние границы включены.

Регион и федеральный округ взяты из `city.csv`. Для одноимённых городов из разных регионов и названий без совпадения оба поля остаются пустыми.

In [59]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE city_reference AS
    SELECT * FROM read_csv(?, all_varchar = true)
""", [str(BASE_DIR / "data/city.csv")])

In [60]:
con.sql("SELECT *  FROM city_reference LIMIT 10").show()

┌───────────────────────────────┬─────────────┬─────────┬──────────────────┬─────────────┬───────────┬───────────┬───────────────┬───────────┬───────────────┬─────────────────┬────────────┬───────────────┬──────────────────────────────────────┬────────────┬────────────────┬─────────────┬─────────────┬────────────┬──────────┬────────────┬────────────┬────────────┬─────────────────┐
│            address            │ postal_code │ country │ federal_district │ region_type │  region   │ area_type │     area      │ city_type │     city      │ settlement_type │ settlement │   kladr_id    │               fias_id                │ fias_level │ capital_marker │    okato    │    oktmo    │ tax_office │ timezone │  geo_lat   │  geo_lon   │ population │ foundation_year │
│            varchar            │   varchar   │ varchar │     varchar      │   varchar   │  varchar  │  varchar  │    varchar    │  varchar  │    varchar    │     varchar     │  varchar   │    varchar    │               varchar     

In [68]:
con.execute(r"""
    CREATE OR REPLACE TEMP VIEW cars_features AS
    WITH city_names AS (
        SELECT coalesce(settlement, city, regexp_extract(address, '(?:^|, )г (.+)$', 1)) AS city_name,
               region, federal_district
        FROM city_reference
    ),
    city_map AS (
        SELECT replace(lower(trim(city_name)), 'ё', 'е') AS city_key,
               min(region) AS region, min(federal_district) AS federal_district
        FROM city_names
        WHERE city_name <> ''
        GROUP BY city_key
        HAVING count(DISTINCT region) = 1 AND count(DISTINCT federal_district) = 1
    )
    SELECT cars.*,
           2023 - year AS car_age,
           round(mileage / nullif(car_age, 0)) AS mileage_per_year,
           CASE
               WHEN brand IN ('Аурус', 'ГАЗ', 'Донинвест', 'ЗИЛ', 'ЗиС', 'ИЖ', 'Лада',
                              'Москвич', 'ТагАЗ', 'УАЗ', 'Derways', 'DW Hower', 'Evolute', 'Vortex')
                   THEN 'Отечественная'
               WHEN brand NOT IN ('Прочие авто', '') THEN 'Иномарка'
           END AS brand_origin,
           CASE
               WHEN price < 1000000 THEN 'Бюджетный'
               WHEN price < 3000000 THEN 'Средний'
               ELSE 'Премиум'
           END AS price_segment,
           city_map.region, city_map.federal_district
    FROM cars_missing_clean AS cars
    LEFT JOIN city_map ON replace(lower(trim(cars.location)), 'ё', 'е') = city_map.city_key
""")

total, with_region = con.execute("SELECT count(*), count(region) FROM cars_features").fetchone()
assert total == rows_after_missing, "После добавления признаков изменилось число объявлений"
print(f"Регион определён для {with_region:,} из {total:,} объявлений ({with_region / total:.1%})" if total else "Выборка пуста")

Регион определён для 485,693 из 521,482 объявлений (93.1%)


In [ ]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW cars_region_clean AS
    SELECT *
    FROM cars_features
    WHERE region IS NOT NULL
""")

rows_after_region = con.sql("SELECT count(*) FROM cars_region_clean").fetchall()[0][0]
print(f"Исключено объявлений без региона: {total - rows_after_region:,}")
print(f"Осталось объявлений: {rows_after_region:,}")

Исключено объявлений без региона: 35,789
Осталось объявлений: 485,693


In [79]:
con.fetchall()

[]

In [66]:
con.sql("SELECT * EXCLUDE (description) FROM cars_region_clean LIMIT 10").show()

┌────────────┬─────────────────┬────────────┬─────────────┬──────────┬───────┬─────────┬──────────────┬───────┬──────────┬─────────────────────┬─────────────────┬───────────────────────────────────────────────────────────────┬─────────────────────┬─────────┬──────────────────┬──────────────┬───────────────┬──────────────────────────────────────────┬──────────────────┐
│   brand    │      name       │  bodyType  │    color    │ fuelType │ year  │ mileage │ transmission │ power │  price   │        date         │    location     │                             link                              │     parse_date      │ car_age │ mileage_per_year │ brand_origin │ price_segment │                  region                  │ federal_district │
│  varchar   │     varchar     │  varchar   │   varchar   │ varchar  │ int16 │  int32  │   varchar    │ int16 │  int64   │      timestamp      │     varchar     │                            varchar                            │      timestamp      │  int16  │

### Определение: дилер или частник

Флаг `is_dealer` отражает признаки дилера в описании объявления. Список регулярных выражений охватывает упоминания собственного салона, приглашения в автоцентр, предложения кредита и обмена.
При совпадении хотя бы с одним шаблоном результат - `TRUE`, при отсутствии совпадений - `FALSE`, при пустом или отсутствующем описании - `NULL`.

In [ ]:
# Определение автосалона
dealer_patterns = [
    # Описание продавца и приглашение в салон
    r"наш(?:ем|его|ему)?\s+(?:автосалон|автоцентр|дилерск(?:ий|ом|ого|ому)\s+центр)(?:е|а|у)?",
    r"(?:мы|являемся)\s*(?:[—–-]\s*)?официальн(?:ый|ым)\s+дилер(?:ом)?",
    r"(?:приглашаем|жд[её]м)\s+вас\s+в\s+(?:наш\s+)?(?:автосалон|автоцентр)",
    r"(?:автосалон|автоцентр)\s+(?:предлагает|представляет)\s+(?:вам\s+)?(?:автомобиль|авто)",
    # Предлагаемые услуги
    r"наши\s+менеджеры\s+(?:проконсультируют|помогут|подберут)",
    r"(?:оформим|оформляем|предлагаем)\s+(?:автокредит|кредитование)",
    r"оформление\s+(?:авто)?кредита\s+(?:в\s+(?:нашем\s+)?автосалоне|на\s+месте)",
    r"принимаем\s+(?:ваш[уи]?\s+)?(?:автомобил[ьи]|авто|машину)\s+в\s+(?:зач[её]т|обмен)",
    r"(?:принимаем|предлагаем|программа|условия\s+по\s+кредиту\s+и)\s+(?:тре[йи]д[\s‐‑–—-]*ин|trade[\s‐‑–—-]*in)",
]
dealer_regex = r"(?:^|[^\p{L}\p{N}])(?:" + "|".join(dealer_patterns) + r")(?:$|[^\p{L}\p{N}])"

con.execute(rf"""
    CREATE OR REPLACE TEMP VIEW cars_seller_flag AS
    SELECT *,
        CASE WHEN regexp_matches(description, '\S') THEN
            regexp_matches(lower(description), '{dealer_regex}')
        END AS is_dealer
    FROM cars_region_clean
""").fetchall()

con.sql("""
    SELECT is_dealer, count(*) AS ads_count
    FROM cars_seller_flag
    GROUP BY is_dealer
    ORDER BY ads_count DESC NULLS LAST
""").show()

┌───────────┬───────────┐
│ is_dealer │ ads_count │
│  boolean  │   int64   │
├───────────┼───────────┤
│ false     │    451917 │
│ true      │     17401 │
│ NULL      │     16375 │
└───────────┴───────────┘



### Количество уникальных описаний объявлений

In [85]:
con.sql("""SELECT count(*), count(DISTINCT description) FROM cars_region_clean""").show()

┌──────────────┬─────────────────────────────┐
│ count_star() │ count(DISTINCT description) │
│    int64     │            int64            │
├──────────────┼─────────────────────────────┤
│       485693 │                      397733 │
└──────────────┴─────────────────────────────┘



## Замена «ё» на «е» и сохранение результата в Parquet

In [ ]:
cars = con.table("cars_seller_flag")
text_columns = [name for name, dtype in zip(cars.columns, cars.types) if dtype == "VARCHAR"]
replacements = [f"replace({name}, 'ё', 'е') AS {name}" for name in text_columns]
clean = cars.project(f"* REPLACE ({', '.join(replacements)})")
clean.write_parquet(str(BASE_DIR / "data/processed/clean.parquet"))